# Tsetlin bake-off on free Colab / Kaggle GPU
`tmu` builds C/CUDA extensions that are painful on Windows. Run the Tsetlin
models here for free instead. Upload `data/features/` (X.parquet, meta.parquet,
features.json) or clone the repo, then run all cells.

In [ ]:
!pip -q install tmu scikit-learn xgboost lightgbm pyarrow pandas pyyaml
# Option A: clone your repo
# !git clone https://github.com/<you>/tsetlin-market-lab.git && cd tsetlin-market-lab
# Option B: upload data/features/ via the Files panel

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

X = pd.read_parquet('data/features/X.parquet')
meta = pd.read_parquet('data/features/meta.parquet')
feat = json.loads(open('data/features/features.json').read())
y = meta['y'].to_numpy(np.uint32)

order = meta.sort_values(['commence_time','match_id','snapshot_ts']).index.to_numpy()
matches = meta.loc[order,'match_id'].drop_duplicates().to_numpy()
n = len(matches); tr_m = matches[:int(n*0.8)]
tr = meta['match_id'].isin(tr_m).to_numpy()
Xtr, ytr = X[feat].to_numpy(np.uint8)[tr], y[tr]
Xte, yte = X[feat].to_numpy(np.uint8)[~tr], y[~tr]
print(Xtr.shape, Xte.shape, 'pos rate', yte.mean().round(3))

In [ ]:
from tmu.models.classification.vanilla_classifier import TMClassifier

tm = TMClassifier(number_of_clauses=500, T=16, s=5.0, weighted_clauses=True, seed=0)
for epoch in range(40):
    tm.fit(Xtr, ytr)
    if epoch % 10 == 0:
        p = tm.predict(Xte)
        print(epoch, 'acc', (p == yte).mean().round(4))

pred = tm.predict(Xte)
print('final acc', (pred == yte).mean().round(4))

In [ ]:
# Dump the most-present clauses as readable rules
n_lit = len(feat)
for cls in (1, 0):
    print('\n=== class', 'MOVE' if cls else 'NO-MOVE', '===')
    shown = 0
    for c in range(tm.number_of_clauses):
        lits = []
        for k in range(n_lit):
            if tm.get_ta_action(clause=c, ta=k, the_class=cls): lits.append(feat[k])
            if tm.get_ta_action(clause=c, ta=k+n_lit, the_class=cls): lits.append('NOT '+feat[k])
        if lits and shown < 15:
            print('IF', ' AND '.join(lits)); shown += 1